[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/traceopt-ai/traceml/blob/main/notebooks/huggingface_dataloading_bottleneck.ipynb)

# Find a data-loading bottleneck with Hugging Face Trainer

A training job can keep making progress while its GPU repeatedly waits for the next batch. This notebook makes that wait visible, then fixes it without rewriting a training loop.

You will train the same Hugging Face ResNet-50 job twice on the real 320px Imagenette dataset. The only difference is three `TrainingArguments` data-loader settings: workers, pinned memory, and persistent workers. For each run, TraceML reports the diagnosis; the final cell compares the saved summaries with `traceml compare`.

The result is intentionally hardware-dependent. More CPU cores can decode images in parallel and keep the GPU fed; a small Colab CPU will still benefit, but may remain input-bound. That is useful information about *your* machine, not a failed demo.

**Before you start:** in Colab, choose **Runtime → Change runtime type → GPU**, then run the cells from top to bottom. This notebook downloads a 326 MiB image archive and a ResNet-50 checkpoint.

## 1. Check that a GPU is available

In [ ]:
!nvidia-smi -L

import torch

print("CUDA available:", torch.cuda.is_available())
assert (
    torch.cuda.is_available()
), "No GPU found. In Colab: Runtime → Change runtime type → GPU, then rerun."

GPU 0: Tesla T4 (UUID: GPU-b3003e28-a103-405a-a1bb-a1d65557997c)
CUDA available: True


## 2. Install the notebook dependencies

TraceML adds `TraceMLTrainerCallback` to the standard Hugging Face `Trainer`. The notebook uses ordinary `TrainingArguments` and `Trainer`; there is no custom training loop to learn.

TraceML is installed with `--no-deps` so it does not replace Colab's NumPy version. Hugging Face packages are installed separately.

In [ ]:
!pip install -q "traceml-ai[hf]" --no-deps
!pip install -q transformers datasets accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 550.1/550.1 kB 19.3 MB/s eta 0:00:00


## 3. Download a compact dataset where image decoding still matters

The official 320px Imagenette variant contains real JPEGs but is much smaller than the full-resolution archive. `num_workers=0` still makes data loading a realistic source of GPU idle time.

In [ ]:
import os

if not os.path.isdir("imagenette2-320/train"):
    !wget -q https://s3.amazonaws.com/fast-ai-imageclas/imagenette2-320.tgz
    !tar -xzf imagenette2-320.tgz

print("CPU cores available:", os.cpu_count())
print(
    "Training images:",
    sum(len(files) for _, _, files in os.walk("imagenette2-320/train")),
)

CPU cores available: 2
Training images: 9469


## 4. The complete Trainer script

This is ordinary Hugging Face training: an AutoModel, `TrainingArguments`, and a `Trainer`-style call to `train()`. The TraceML-specific pieces are deliberately small:

1. Call `traceml_hf.init()` once so TraceML can measure internally-created DataLoader fetches.
2. Add `traceml_hf.TraceMLTrainerCallback()` to the standard `Trainer` callbacks.

The `--profile` flag is the experiment switch. Model, images, batch size, augmentation, and number of steps stay fixed.

In [ ]:
%%writefile hf_train.py
import argparse
import os

import torch
from torch.utils.data import Dataset
from torchvision.datasets import ImageFolder
from torchvision.transforms import Compose, Normalize, RandomHorizontalFlip, RandomResizedCrop, ToTensor
from transformers import AutoImageProcessor, AutoModelForImageClassification, DefaultDataCollator, Trainer, TrainingArguments

from traceml_ai.integrations import huggingface as traceml_hf

MODEL_NAME = "microsoft/resnet-50"


class ImagenetteForTrainer(Dataset):
    """Return the field names expected by AutoModelForImageClassification."""

    def __init__(self, root, transform):
        self.images = ImageFolder(root, transform=transform)
        self.classes = self.images.classes

    def __len__(self):
        return len(self.images)

    def __getitem__(self, index):
        pixel_values, label = self.images[index]
        return {"pixel_values": pixel_values, "labels": label}


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--profile", choices=["baseline", "optimized"], required=True)
    parser.add_argument("--data-dir", default="imagenette2-320")
    parser.add_argument("--batch-size", type=int, default=32)
    parser.add_argument("--max-steps", type=int, default=200)
    args = parser.parse_args()

    torch.manual_seed(42)
    optimized = args.profile == "optimized"
    # The only experimental change: DataLoader settings exposed by TrainingArguments.
    num_workers = min(4, os.cpu_count() or 2) if optimized else 0

    image_processor = AutoImageProcessor.from_pretrained(MODEL_NAME)
    crop_size = image_processor.size.get("height", image_processor.size.get("shortest_edge", 224))
    transform = Compose([
        RandomResizedCrop(crop_size),
        RandomHorizontalFlip(),
        ToTensor(),
        Normalize(mean=image_processor.image_mean, std=image_processor.image_std),
    ])
    train_dataset = ImagenetteForTrainer(
        os.path.join(args.data_dir, "train"), transform=transform
    )

    id2label = {index: name for index, name in enumerate(train_dataset.classes)}
    label2id = {name: index for index, name in id2label.items()}
    model = AutoModelForImageClassification.from_pretrained(
        MODEL_NAME,
        num_labels=len(train_dataset.classes),
        id2label=id2label,
        label2id=label2id,
        ignore_mismatched_sizes=True,
    )

    training_args = TrainingArguments(
        output_dir=f"outputs/{args.profile}",
        per_device_train_batch_size=args.batch_size,
        max_steps=args.max_steps,
        learning_rate=1e-4,
        save_strategy="no",
        report_to="none",
        disable_tqdm=True,
        remove_unused_columns=False,
        dataloader_num_workers=num_workers,
        dataloader_pin_memory=optimized,
        dataloader_persistent_workers=optimized and num_workers > 0,
    )

    print(
        f"[demo] profile={args.profile} workers={num_workers} "
        f"pin_memory={optimized} persistent_workers={optimized and num_workers > 0} "
        f"batch_size={args.batch_size} max_steps={args.max_steps}",
        flush=True,
    )

    traceml_hf.init()  # Required once: enables DataLoader fetch and step timing.
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        data_collator=DefaultDataCollator(),
        callbacks=[traceml_hf.TraceMLTrainerCallback()],
    )
    trainer.train()


if __name__ == "__main__":
    main()

Writing hf_train.py


## 5. Run the baseline: one process loads every image

With `dataloader_num_workers=0`, the main training process decodes and augments each batch itself. TraceML should report `INPUT-BOUND` when that work leaves the GPU waiting. The run uses 200 steps—enough for a stable diagnosis without turning this into a long training job.

In [ ]:
!traceml run --mode summary --logs-dir logs --run-name hf_baseline hf_train.py --args --profile baseline --data-dir imagenette2-320 --max-steps 200 --batch-size 32

[TraceML] Starting aggregator on 127.0.0.1:29765 (connect=127.0.0.1, ui=summary, profile=run)
[TraceML] Launching TraceML aggregator: /usr/bin/python3 /usr/local/lib/python3.12/dist-packages/traceml_ai/aggregator/aggregator_main.py
[TraceML] Aggregator PID: 1529
[TraceML] Aggregator ready on 127.0.0.1:29765 (workers connect to 127.0.0.1:29765, session=hf_baseline, ui=summary). Press Ctrl+C to stop.
[TraceML] Aggregator ready.
[TraceML] Launching training process: /usr/bin/python3 -m torch.distributed.run --nnodes=1 --nproc_per_node=1 --node_rank=0 --master_addr=127.0.0.1 --master_port=29500 /usr/local/lib/python3.12/dist-packages/traceml_ai/runtime/executor.py -- --profile baseline --data-dir imagenette2-320 --max-steps 200 --batch-size 32
preprocessor_config.json: 100% 266/266 [00:00<00:00, 1.30MB/s]
config.json: 100% 69.6k/69.6k [00:00<00:00, 19.6MB/s]
[transformers] You passed `num_labels=10` which is incompatible to the `id2label` map of length `1000`.

model.safetensors: downloadi

## 6. Run the optimized loader

Same images, model, batch size, and steps. This profile lets up to four CPU workers decode ahead, pins batches for faster transfer, and keeps workers alive after the loader is created. If your CPU can hide the decode work, the verdict can flip to `COMPUTE-BOUND`.

In [ ]:
!traceml run --mode summary --logs-dir logs --run-name hf_optimized hf_train.py --args --profile optimized --data-dir imagenette2-320 --max-steps 200 --batch-size 32

[TraceML] Starting aggregator on 127.0.0.1:29765 (connect=127.0.0.1, ui=summary, profile=run)
[TraceML] Launching TraceML aggregator: /usr/bin/python3 /usr/local/lib/python3.12/dist-packages/traceml_ai/aggregator/aggregator_main.py
[TraceML] Aggregator PID: 2235
[TraceML] Aggregator ready on 127.0.0.1:29765 (workers connect to 127.0.0.1:29765, session=hf_optimized, ui=summary). Press Ctrl+C to stop.
[TraceML] Aggregator ready.
[TraceML] Launching training process: /usr/bin/python3 -m torch.distributed.run --nnodes=1 --nproc_per_node=1 --node_rank=0 --master_addr=127.0.0.1 --master_port=29500 /usr/local/lib/python3.12/dist-packages/traceml_ai/runtime/executor.py -- --profile optimized --data-dir imagenette2-320 --max-steps 200 --batch-size 32
[transformers] You passed `num_labels=10` which is incompatible to the `id2label` map of length `1000`.
Loading weights: 100% 320/320 [00:00<00:00, 6624.27it/s]
[transformers] ResNetForImageClassification LOAD REPORT from: microsoft/resnet-50
Key  

## 7. Compare the two runs

TraceML writes a portable `final_summary.json` for each run. `traceml compare` prints a compact comparison and writes JSON and text artifacts for the baseline-versus-optimized result.

In [ ]:
!traceml compare logs/hf_baseline/final_summary.json logs/hf_optimized/final_summary.json --output=logs/hf_baseline_vs_optimized

+--------------------------------------------------------------------------------------+
|  TraceML Compare                                                                     |
+--------------------------------------------------------------------------------------+
|                                                                                      |
|  A: hf_baseline                                                                      |
|  B: hf_optimized                                                                     |
|  Delta: B - A                                                                        |
|  Primary diagnosis: INPUT-BOUND -> COMPUTE-BOUND (changed)                           |
|                                                                                      |
|  Verdict: IMPROVEMENT                                                                |
|  Why: Step time decreased by 24.5%.                                                  |
|                    

## Use this in your own Trainer

The experiment is a pattern, not a special ResNet trick. In an existing Hugging Face script, call `traceml_hf.init()` once and add `traceml_hf.TraceMLTrainerCallback()` to the standard `Trainer` callbacks. Then run it through `traceml run --mode summary ...`.

If TraceML says `INPUT-BOUND`, start by testing `dataloader_num_workers`, `dataloader_pin_memory`, and `dataloader_persistent_workers` one change at a time. Keep the change only when the measured wall time and input wait improve on the hardware that will actually run your job.